# Data Discovery

Search and retrieve observational data from the **SRCNet Data Discovery TAP service**
using `astroquery.srcnet`.

```
DataDiscovery
  ├─▶ .get_tables()                        # list all TAP tables
  ├─▶ .get_columns(table)                  # inspect column definitions
  ├─▶ .get_collections()                   # data collections + counts
  ├─▶ .query(adql)                         # raw ADQL
  ├─▶ .query_region(coord, radius)         # cone search
  ├─▶ .query_name(name)                    # target name search
  ├─▶ .query_observations(…)              # keyword-filtered observations
  ├─▶ .get_artifacts(obs_id)              # files for an observation
  ├─▶ .nl_to_adql(text)                   # NL → ADQL translation
  └─▶ .query_natural(text)                # NL → ADQL + execute
```

The service follows the **CAOM2** data model.  All methods return
`astropy.table.Table` objects and render as rich HTML in Jupyter.

## 1 · Import

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

from astroquery.srcnet import DataDiscovery, SRCNet

## 2 · Explore the schema

Before querying, inspect what tables and columns the service exposes.

In [ ]:
# All available tables
DataDiscovery.get_tables()

In [ ]:
# Columns in the main Observations table
DataDiscovery.get_columns("caom2.Observation")

In [ ]:
# Columns in the Plane table (data products with spatial/spectral coverage)
DataDiscovery.get_columns("caom2.Plane")

## 3 · Data collections

Each top-level collection groups observations from a single telescope or survey.

In [ ]:
DataDiscovery.get_collections()

## 4 · Spatial cone search

Find all observations whose pointing falls within a circle on the sky.  
Coordinates are passed as an `astropy.coordinates.SkyCoord`,
radius as an `astropy.units.Quantity`.

In [ ]:
# Search around the Orion Nebula (M42)
orion = SkyCoord(83.8221, -5.3911, unit="deg")

results = DataDiscovery.query_region(orion, radius=1.0 * u.deg)
print(f"{len(results)} observations found")
results

In [ ]:
# Narrow to a specific collection
DataDiscovery.query_region(orion, radius=1.0 * u.deg, collection="JCMT")

## 5 · Search by target name

`query_name` does a case-insensitive substring match on `target_name`.

In [ ]:
DataDiscovery.query_name("Crab")

In [ ]:
# Restrict to a specific collection
DataDiscovery.query_name("M31", collection="CFHT")

## 6 · Filter observations by telescope / instrument

`query_observations` provides keyword filters without writing ADQL.

In [ ]:
# All SCUBA-2 observations
DataDiscovery.query_observations(collection="JCMT", instrument="SCUBA-2")

In [ ]:
# Observations targeting anything with "Orion" in the name
DataDiscovery.query_observations(target_name="Orion", maxrec=20)

## 7 · Retrieve file artifacts

Once you have an `observationID`, `get_artifacts` returns all associated files
with their URIs, types, and sizes.

In [ ]:
# Replace with a real observationID from your results above
obs_id = "scuba2_00001_20230101T000000"

DataDiscovery.get_artifacts(obs_id)

## 8 · Raw ADQL queries

For queries not covered by the convenience methods, use `DataDiscovery.query(adql)` directly.  
The underlying service is IVOA TAP — any valid ADQL is accepted.

In [ ]:
# Observation count per telescope
DataDiscovery.query("""
    SELECT telescope_name, COUNT(*) AS n
    FROM caom2.Observation
    GROUP BY telescope_name
    ORDER BY n DESC
""")

In [ ]:
# Observations with their spatial footprint (joined Observation + Plane)
DataDiscovery.query("""
    SELECT TOP 20
        o.observationID,
        o.collection,
        o.target_name,
        p.position_center_ra  AS ra,
        p.position_center_dec AS dec,
        p.energy_bandpassName AS bandpass,
        p.time_exposure       AS t_exp
    FROM caom2.Observation AS o
    JOIN caom2.Plane        AS p ON o.obsID = p.obsID
    WHERE p.position_center_ra IS NOT NULL
    ORDER BY o.collection, o.observationID
""")

In [ ]:
# Data volume per bandpass
DataDiscovery.query("""
    SELECT p.energy_bandpassName AS bandpass,
           COUNT(*)               AS n_planes,
           SUM(a.contentLength)   AS total_bytes
    FROM caom2.Plane    AS p
    JOIN caom2.Artifact AS a ON p.planeID = a.planeID
    WHERE p.energy_bandpassName IS NOT NULL
    GROUP BY p.energy_bandpassName
    ORDER BY total_bytes DESC
""")

## 9 · Natural language → ADQL

`nl_to_adql()` translates a plain-English question into ADQL using the SRCNet
remote chat service — no local model setup required.

`query_natural()` does the same translation and also executes the resulting query.

In [ ]:
# Translate only — inspect the ADQL before running
adql = DataDiscovery.nl_to_adql("how many observations are there per collection?")
print(adql)

In [ ]:
# Translate and execute
adql, results = DataDiscovery.query_natural(
    "show the 10 most recent JCMT observations with their target name and exposure time",
    verbose=True,   # prints the generated ADQL
)
results

In [ ]:
# Spatial query expressed in natural language
adql, results = DataDiscovery.query_natural(
    "find all observations within 1 degree of RA=83.8, Dec=-5.4",
    verbose=True,
)
results

In [ ]:
# Aggregate query
adql, results = DataDiscovery.query_natural(
    "count science observations grouped by instrument",
    verbose=True,
)
results

## 10 · Conversational interface

For multi-turn questions mixing data discovery and software discovery,
use the chat — it preserves context across calls.

In [ ]:
# Print usage examples
SRCNet.chat()

In [ ]:
t = SRCNet.chat("How many JCMT observations are there in total?")

In [ ]:
# Follow-up — context is preserved
t = SRCNet.chat("Break that down by instrument")

In [ ]:
# Start a fresh session
SRCNet._chat.reset()